# LeetCode 817 — Linked List Components bằng Python

Link đề: https://leetcode.com/problems/linked-list-components/ · Độ khó: **Medium** · Chủ đề: Hash Table, Linked List

Skeleton LeetCode:

```python
# Definition for singly-linked list.
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def numComponents(self, head: Optional[ListNode], nums: List[int]) -> int:
        ...
```

Ghi chú: trên LeetCode lớp `ListNode` được cho sẵn và danh sách liên kết được truyền vào dưới dạng một mảng giá trị. Trong notebook này ta **tự dựng** `ListNode`, hàm dựng danh sách từ mảng (`build_list`) và hàm đọc ngược ra mảng (`list_to_array`) để chạy mọi ví dụ ngay tại đây.

Bài này thực chất là đếm "đoạn liên tiếp" (run) trong một dãy, nơi mỗi phần tử được gắn nhãn **thuộc / không thuộc** tập `nums`. Nó dạy ba thứ: duyệt danh sách liên kết, dùng `set` để tra cứu `O(1)`, và đếm số đoạn bằng **cờ trạng thái** hoặc bằng công thức "số nút trừ số cạnh".

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho danh sách liên kết các giá trị **đôi một khác nhau** và một tập `nums` là tập con của các giá trị đó; đếm số **thành phần liên thông** (đoạn nút **liên tiếp** trong danh sách mà mọi nút đều có giá trị thuộc `nums`, và đoạn là **tối đại**).
2. Nắm các công cụ Python cần dùng: lớp `ListNode`, vòng `while head:`, `set` và phép `in` (`O(1)`) so với `in list` (`O(m)`), biến cờ `in_run`, mảng `bool` đánh chỉ số theo giá trị, `itertools.groupby`.
3. Đi từ **duyệt + `in` trên danh sách (`O(n·m)`)** → **`set` + đếm điểm kết thúc đoạn** → **mảng `bool` theo giá trị** → **công thức `số nút - số cạnh`**.
4. Hiểu mấu chốt: một thành phần là một **đường đi** (path) các nút liên tiếp; số thành phần bằng số **điểm kết thúc** của các đoạn, hay bằng `số nút thuộc nums - số cặp nút liên tiếp cùng thuộc nums`.
5. Tránh các lỗi kinh điển: đếm số nút thuộc `nums` thay vì số đoạn, lỗi `AttributeError` khi `head.next` là `None`, quên đặt lại cờ khi gặp nút không thuộc `nums`, và nhầm "liên tiếp trong danh sách" với "liên tiếp về giá trị".

## 2. Đề bài & ví dụ

Cho `head` của một danh sách liên kết gồm các giá trị **nguyên khác nhau**, và một mảng số nguyên `nums` là **tập con** các giá trị của danh sách.

Trả về **số thành phần liên thông** của `nums`: một thành phần là một dãy **không rỗng, tối đại**, gồm các nút **liên tiếp** trong danh sách sao cho giá trị của mọi nút trong dãy đều thuộc `nums`.

**Ví dụ 1**

- Input: `head = [0,1,2,3]`, `nums = [0,1,3]`
- Output: `2`
- Giải thích: `0` và `1` liên tiếp nên thành `[0, 1]`; `3` đứng riêng thành `[3]`. Có `2` thành phần.

**Ví dụ 2**

- Input: `head = [0,1,2,3,4]`, `nums = [0,3,1,4]`
- Output: `2`
- Giải thích: `[0, 1]` và `[3, 4]`.

**Ràng buộc:**

- Số nút `n` thỏa `1 <= n <= 10^4`.
- `0 <= Node.val < n`, mọi `Node.val` khác nhau.
- `1 <= nums.length <= n`, `0 <= nums[i] < n`, mọi giá trị của `nums` khác nhau.

Lưu ý: "liên tiếp" nói về **vị trí trong danh sách**, không phải về độ lớn của giá trị. Danh sách có thể là hoán vị bất kỳ, ví dụ `[3, 1, 0, 2, 4]`.

In [1]:
from typing import Optional, List
from itertools import groupby
import random
import time


class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def build_list(values):
    # dựng danh sách liên kết từ mảng; trả về head (None nếu mảng rỗng)
    dummy = ListNode()
    cur = dummy
    for v in values:
        cur.next = ListNode(v)
        cur = cur.next
    return dummy.next


def list_to_array(head):
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out


examples = [
    ([0, 1, 2, 3], [0, 1, 3], 2),
    ([0, 1, 2, 3, 4], [0, 3, 1, 4], 2),
]
for vals, nums, expected in examples:
    assert list_to_array(build_list(vals)) == vals
    print("head =", vals, "| nums =", nums, "-> kỳ vọng", expected)

head = [0, 1, 2, 3] | nums = [0, 1, 3] -> kỳ vọng 2
head = [0, 1, 2, 3, 4] | nums = [0, 3, 1, 4] -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. `ListNode` và duyệt danh sách liên kết

Mỗi nút có `val` (giá trị) và `next` (nút kế tiếp, `None` nếu là nút cuối). Ta duyệt bằng vòng `while head:` (vì `None` là "sai"), mỗi vòng xử lý `head.val` rồi đi tiếp `head = head.next`. Hàm `build_list` ở trên dùng nút giả `dummy` để không phải xử lý riêng nút đầu.

In [2]:
head = build_list([5, 6, 7])
print(list_to_array(head))
cur = head
while cur:
    print("nút", cur.val, "| next =", cur.next.val if cur.next else None)
    cur = cur.next

[5, 6, 7]
nút 5 | next = 6
nút 6 | next = 7
nút 7 | next = None


### 3.2. `set` và phép `in`

`x in danh_sách` phải so sánh lần lượt: `O(độ dài)`. `x in set` dùng bảng băm: `O(1)` trung bình. Vì ta sẽ hỏi "giá trị này có thuộc `nums` không?" cho **mỗi nút**, đổi `nums` sang `set` một lần (`O(m)`) rồi tra cứu `O(1)` sẽ nhanh hơn rõ rệt.

In [3]:
nums = [0, 1, 3]
s = set(nums)
print(2 in nums, 2 in s, 3 in s)

big = list(range(100000))
bs = set(big)
t0 = time.perf_counter(); [99999 in big for _ in range(200)]; t1 = time.perf_counter() - t0
t0 = time.perf_counter(); [99999 in bs for _ in range(200)]; t2 = time.perf_counter() - t0
print(f"200 lần tra: list {t1:.4f}s | set {t2:.6f}s")

False False True
200 lần tra: list 0.1691s | set 0.000095s


### 3.3. Đếm số "đoạn" bằng cờ trạng thái

Bài toán "đếm số đoạn liên tiếp các phần tử thỏa điều kiện" có mẫu chung: duyệt dãy, giữ biến cờ `in_run` cho biết **phần tử trước có thuộc một đoạn đang mở không**.

- Gặp phần tử **thỏa** mà `in_run` là `False`: bắt đầu một đoạn mới, `count += 1`, `in_run = True`.
- Gặp phần tử **thỏa** mà `in_run` là `True`: đoạn đang tiếp diễn, không làm gì.
- Gặp phần tử **không thỏa**: đoạn kết thúc, `in_run = False`.

In [4]:
flags = [True, True, False, True, False, False, True, True, True]
count, in_run = 0, False
for f in flags:
    if f and not in_run:
        count += 1
    in_run = f
print("số đoạn True liên tiếp:", count)    # 3

số đoạn True liên tiếp: 3


### 3.4. `itertools.groupby` (dùng cho oracle kiểm thử)

`groupby(iterable, key)` gom các phần tử **liên tiếp** có cùng `key`. Ta dùng nó để viết một lời giải kiểm tra khác hẳn: chuyển danh sách thành mảng, đánh nhãn `True/False`, rồi đếm số nhóm `True`.

In [5]:
flags = [True, True, False, True, False, False, True, True, True]
groups = [(k, len(list(g))) for k, g in groupby(flags)]
print(groups)
print("số nhóm True:", sum(1 for k, _ in groups if k))

[(True, 2), (False, 1), (True, 1), (False, 2), (True, 3)]
số nhóm True: 3


### 3.5. Mảng `bool` đánh chỉ số theo giá trị

Vì `0 <= Node.val < n`, ta có thể dùng mảng `mark` với `mark[v] = True` nếu `v` thuộc `nums`. Tra cứu bằng chỉ số `mark[v]` cũng `O(1)` và không cần băm; đổi lại cần biết giới hạn giá trị.

In [6]:
nums = [0, 3, 1, 4]
mark = [False] * (max(nums) + 1)
for v in nums:
    mark[v] = True
print(mark)
print([mark[v] if v < len(mark) else False for v in [0, 1, 2, 3, 4, 7]])

[True, True, False, True, True]
[True, True, False, True, True, False]


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Lấy ví dụ 1: `head = [0,1,2,3]`, `nums = [0,1,3]`. Đánh nhãn từng nút: thuộc `nums` (T) hay không (F):

| Vị trí | Giá trị | Nhãn |
|---|---|---|
| 0 | 0 | T |
| 1 | 1 | T |
| 2 | 2 | F |
| 3 | 3 | T |

Dãy nhãn: `T T F T`. Các đoạn `T` liên tiếp: `[T T]` và `[T]`. Có `2` đoạn. Mỗi lần gặp `F` thì đoạn kết thúc.

Để chắc ý nghĩa của "liên tiếp theo vị trí", xét `head = [3, 1, 0, 2, 4]`, `nums = [1, 0, 4, 3]`: nhãn `T T T F T` (vì `3, 1, 0` đều thuộc `nums`, `2` thì không, `4` thuộc). Hai đoạn: `[3, 1, 0]` và `[4]`. Chú ý `3` và `1` **không liên tiếp về giá trị** nhưng vẫn liên tiếp trong danh sách nên cùng một thành phần.

Trực giác: **chuyển danh sách thành dãy nhãn `T/F` rồi đếm số khối `T`**. Số khối `T` bằng số lần "chuyển từ `F` (hoặc từ đầu danh sách) sang `T`", cũng bằng số lần "chuyển từ `T` sang `F` (hoặc kết thúc danh sách)".

In [7]:
def labels(head, nums):
    s = set(nums)
    return ["T" if v in s else "F" for v in list_to_array(head)]

print(labels(build_list([0, 1, 2, 3]), [0, 1, 3]))
print(labels(build_list([3, 1, 0, 2, 4]), [1, 0, 4, 3]))

['T', 'T', 'F', 'T']
['T', 'T', 'T', 'F', 'T']


## 5. Cách 1 — Brute Force: dùng `in` trên danh sách `nums`

### Ý tưởng

Làm đúng theo trực giác nhưng không chuẩn bị gì thêm: duyệt từng nút, kiểm tra `head.val in nums` với `nums` là **danh sách** (mỗi lần `O(m)`), và dùng cờ `in_run` đếm số đoạn.

### Code

In [8]:
def comp_bruteforce(head, nums):
    ans = 0
    in_run = False
    while head:
        if head.val in nums:                # nums là list: mỗi lần kiểm tra tốn O(m)
            if not in_run:
                ans += 1
                in_run = True
        else:
            in_run = False
        head = head.next
    return ans

for vals, nums, expected in examples:
    print(vals, nums, "->", comp_bruteforce(build_list(vals), nums), "(kỳ vọng", expected, ")")

[0, 1, 2, 3] [0, 1, 3] -> 2 (kỳ vọng 2 )
[0, 1, 2, 3, 4] [0, 3, 1, 4] -> 2 (kỳ vọng 2 )


### Độ phức tạp và nút thắt

- Time: `n` nút, mỗi nút `O(m)` để tra `in list`: `O(n · m)`. Với `n = m = 10^4` là hàng chục triệu phép so sánh.
- Space: `O(1)`.

**Nút thắt:** thao tác "có thuộc `nums` không" lặp đi lặp lại trên một danh sách không có cấu trúc. Hãy đo thời gian khi `n` tăng.

In [9]:
rng = random.Random(1)
for n in [500, 1000, 2000, 4000]:
    vals = list(range(n)); rng.shuffle(vals)
    nums = rng.sample(range(n), n // 2)
    head = build_list(vals)
    t0 = time.perf_counter()
    comp_bruteforce(head, nums)
    print(f"n={n}: {time.perf_counter() - t0:.4f}s")

n=500: 0.0010s
n=1000: 0.0038s
n=2000: 0.0128s


n=4000: 0.0403s


Thời gian tăng gấp khoảng 4 lần mỗi khi `n` gấp đôi (vì cả `n` và `m` cùng tăng gấp đôi): đúng `O(n · m)`. Với `n = 10^4` cách này vẫn qua được giới hạn của đề (khoảng vài phần mười giây, chậm hơn cách dùng `set` cả trăm lần ở mục 11), nhưng thuật toán `O(n · m)` sẽ sụp đổ khi dữ liệu lớn hơn.

## 6. Ý tưởng thuật toán / bước đột phá

Nút thắt nằm ở phép tra cứu. Bước đột phá rất đơn giản: **đổi `nums` thành `set` một lần**, rồi mỗi lần tra cứu chỉ `O(1)`. Thời gian toàn bài còn `O(n + m)`.

Ngoài ra ta có thể đếm số đoạn **không cần cờ**, bằng cách đếm **điểm kết thúc**: một nút `v` thuộc `nums` là **cuối đoạn** nếu nút kế tiếp **không tồn tại** hoặc **không thuộc `nums`**. Mỗi đoạn có đúng một nút cuối đoạn, nên:

> số thành phần = số nút `v` thuộc `nums` mà (`v.next` là `None` hoặc `v.next.val` không thuộc `nums`).

Hãy kiểm tra trên ví dụ 1 (`T T F T`): nút `1` (nhãn `T`, kế tiếp là `2` thuộc `F`) là cuối đoạn; nút `3` (nhãn `T`, không còn nút sau) là cuối đoạn; nút `0` có kế tiếp `1` thuộc `T` nên không phải cuối đoạn. Có `2` điểm kết thúc: đúng.

In [10]:
def end_of_run_nodes(head, nums):
    s = set(nums)
    ends = []
    while head:
        if head.val in s and (head.next is None or head.next.val not in s):
            ends.append(head.val)
        head = head.next
    return ends

print(end_of_run_nodes(build_list([0, 1, 2, 3]), [0, 1, 3]))        # [1, 3]
print(end_of_run_nodes(build_list([3, 1, 0, 2, 4]), [1, 0, 4, 3]))  # [0, 4]

[1, 3]
[0, 4]


## 7. Các cách tiếp theo

Ta có 4 cách khác nhau thực sự:

| Cách | Ý tưởng |
|---|---|
| 1 | Duyệt + `in` trên danh sách `nums`, cờ `in_run` (mục 5) |
| 2 | **`set`** + đếm **nút cuối đoạn** (cách nộp) |
| 3 | **Mảng `bool` theo giá trị** + cờ `prev` (không dùng `set`) |
| 4 | **Công thức `số nút - số cạnh`**: mỗi đoạn là đường đi có `k` nút và `k - 1` cạnh |

### Cách 2 — `set` và đếm nút cuối đoạn

**Giải thích từng đoạn:**

- `s = set(nums)`: tra cứu `O(1)`.
- Duyệt từng nút `head`: nếu `head.val` thuộc `s` **và** (nút sau không tồn tại **hoặc** giá trị nút sau không thuộc `s`) thì đây là nút cuối của một đoạn: `ans += 1`.
- Thứ tự của hai điều kiện trong `or` quan trọng: kiểm tra `head.next is None` **trước** để không truy cập `head.next.val` khi `head.next` là `None`.

### Code demo

In [11]:
def comp_set(head, nums):
    s = set(nums)
    ans = 0
    while head:
        if head.val in s and (head.next is None or head.next.val not in s):
            ans += 1
        head = head.next
    return ans

for vals, nums, expected in examples:
    print(vals, nums, "->", comp_set(build_list(vals), nums), "(kỳ vọng", expected, ")")

[0, 1, 2, 3] [0, 1, 3] -> 2 (kỳ vọng 2 )
[0, 1, 2, 3, 4] [0, 3, 1, 4] -> 2 (kỳ vọng 2 )


### Code nộp LeetCode (Cách 2)

```python
class Solution:
    def numComponents(self, head: Optional[ListNode], nums: List[int]) -> int:
        s = set(nums)
        ans = 0
        while head:
            if head.val in s and (head.next is None or head.next.val not in s):
                ans += 1
            head = head.next
        return ans
```

### Độ phức tạp

- Time: `O(n + m)`: tạo `set` rồi duyệt danh sách một lần.
- Space: `O(m)` cho `set`.

### Cách 3 — Mảng `bool` theo giá trị và cờ `prev`

**Ý tưởng khác biệt:** thay `set` bằng mảng `mark` với `mark[v] = True` nếu `v` thuộc `nums` (nhờ `0 <= val < n`). Duyệt danh sách giữ cờ `prev` = nút **trước** có thuộc `nums` không. Mỗi nút thuộc `nums` mà `prev` là `False` là **đầu** của một đoạn mới. (Đếm đầu đoạn thay vì cuối đoạn: hai cách đếm cho cùng kết quả.)

### Code demo

In [12]:
def comp_array(head, nums):
    mark = [False] * (max(nums) + 1)
    for v in nums:
        mark[v] = True
    ans = 0
    prev = False
    while head:
        cur = head.val < len(mark) and mark[head.val]       # giá trị lớn hơn max(nums) chắc chắn không thuộc nums
        if cur and not prev:
            ans += 1                                         # đầu một đoạn mới
        prev = cur
        head = head.next
    return ans

for vals, nums, expected in examples:
    print(vals, nums, "->", comp_array(build_list(vals), nums), "(kỳ vọng", expected, ")")

[0, 1, 2, 3] [0, 1, 3] -> 2 (kỳ vọng 2 )
[0, 1, 2, 3, 4] [0, 3, 1, 4] -> 2 (kỳ vọng 2 )


### Code nộp LeetCode (Cách 3)

```python
class Solution:
    def numComponents(self, head: Optional[ListNode], nums: List[int]) -> int:
        mark = [False] * (max(nums) + 1)
        for v in nums:
            mark[v] = True
        ans = 0
        prev = False
        while head:
            cur = head.val < len(mark) and mark[head.val]
            if cur and not prev:
                ans += 1
            prev = cur
            head = head.next
        return ans
```

### Độ phức tạp

- Time: `O(n + m)`.
- Space: `O(max(nums))` = `O(n)` cho mảng `mark`.

### Cách 4 — Công thức `số nút - số cạnh`

**Ý tưởng khác biệt:** xem các nút thuộc `nums` như đỉnh, nối hai nút **liên tiếp** trong danh sách khi **cả hai** đều thuộc `nums` bằng một cạnh. Đồ thị thu được là các **đường đi rời nhau**; mỗi thành phần là một đường đi có `k` nút và `k - 1` cạnh. Cộng lại: `Σ nút - Σ cạnh = Σ (k - (k - 1)) = số thành phần`.

Vậy đáp án = `số nút thuộc nums - số cặp nút liên tiếp cùng thuộc nums`.

### Code demo

In [13]:
def comp_formula(head, nums):
    s = set(nums)
    nodes = edges = 0
    while head:
        if head.val in s:
            nodes += 1
            if head.next is not None and head.next.val in s:
                edges += 1
        head = head.next
    return nodes - edges

for vals, nums, expected in examples:
    print(vals, nums, "->", comp_formula(build_list(vals), nums), "(kỳ vọng", expected, ")")

[0, 1, 2, 3] [0, 1, 3] -> 2 (kỳ vọng 2 )
[0, 1, 2, 3, 4] [0, 3, 1, 4] -> 2 (kỳ vọng 2 )


### Code nộp LeetCode (Cách 4)

```python
class Solution:
    def numComponents(self, head: Optional[ListNode], nums: List[int]) -> int:
        s = set(nums)
        nodes = edges = 0
        while head:
            if head.val in s:
                nodes += 1
                if head.next is not None and head.next.val in s:
                    edges += 1
            head = head.next
        return nodes - edges
```

### Độ phức tạp

- Time: `O(n + m)`.
- Space: `O(m)` cho `set`.

## 8. TRACE từng bước

Hàm `trace_components(head, nums)` in từng nút: giá trị, có thuộc `nums` không, nhãn của nút kế tiếp, có phải nút cuối đoạn không và số đoạn đã đếm.

Bảng trạng thái cho **ví dụ 1** (`head = [0,1,2,3]`, `nums = [0,1,3]`) bằng Cách 2:

| Nút | Thuộc `nums`? | Nút sau | Nút sau thuộc `nums`? | Là cuối đoạn? | `ans` |
|---|---|---|---|---|---|
| 0 | có | 1 | có | không | 0 |
| 1 | có | 2 | không | **có** | 1 |
| 2 | không | 3 | - | - | 1 |
| 3 | có | không có | - | **có** | 2 |

In [14]:
def trace_components(head, nums):
    s = set(nums)
    ans = 0
    while head:
        inside = head.val in s
        nxt = head.next
        nxt_txt = "không có" if nxt is None else f"{nxt.val} ({'thuộc' if nxt.val in s else 'không thuộc'})"
        if not inside:
            print(f"nút {head.val}: không thuộc nums -> bỏ qua (nút sau: {nxt_txt})")
        else:
            is_end = nxt is None or nxt.val not in s
            if is_end:
                ans += 1
            print(f"nút {head.val}: thuộc nums | nút sau: {nxt_txt} -> {'CUỐI ĐOẠN, ans += 1' if is_end else 'đoạn còn tiếp'} | ans = {ans}")
        head = nxt
    return ans

print("=>", trace_components(build_list([0, 1, 2, 3]), [0, 1, 3]))

nút 0: thuộc nums | nút sau: 1 (thuộc) -> đoạn còn tiếp | ans = 0
nút 1: thuộc nums | nút sau: 2 (không thuộc) -> CUỐI ĐOẠN, ans += 1 | ans = 1
nút 2: không thuộc nums -> bỏ qua (nút sau: 3 (thuộc))
nút 3: thuộc nums | nút sau: không có -> CUỐI ĐOẠN, ans += 1 | ans = 2
=> 2


In [15]:
print("=>", trace_components(build_list([3, 1, 0, 2, 4]), [1, 0, 4, 3]))
print()
print("=>", trace_components(build_list([0, 1, 2, 3, 4]), [0, 3, 1, 4]))

nút 3: thuộc nums | nút sau: 1 (thuộc) -> đoạn còn tiếp | ans = 0
nút 1: thuộc nums | nút sau: 0 (thuộc) -> đoạn còn tiếp | ans = 0
nút 0: thuộc nums | nút sau: 2 (không thuộc) -> CUỐI ĐOẠN, ans += 1 | ans = 1
nút 2: không thuộc nums -> bỏ qua (nút sau: 4 (thuộc))
nút 4: thuộc nums | nút sau: không có -> CUỐI ĐOẠN, ans += 1 | ans = 2
=> 2

nút 0: thuộc nums | nút sau: 1 (thuộc) -> đoạn còn tiếp | ans = 0
nút 1: thuộc nums | nút sau: 2 (không thuộc) -> CUỐI ĐOẠN, ans += 1 | ans = 1
nút 2: không thuộc nums -> bỏ qua (nút sau: 3 (thuộc))
nút 3: thuộc nums | nút sau: 4 (thuộc) -> đoạn còn tiếp | ans = 1
nút 4: thuộc nums | nút sau: không có -> CUỐI ĐOẠN, ans += 1 | ans = 2
=> 2


Ở ví dụ thứ hai (`[3, 1, 0, 2, 4]`): nút `3` kế tiếp `1` (thuộc) nên chưa kết thúc; `1` kế tiếp `0` (thuộc) cũng chưa; `0` kế tiếp `2` (không thuộc) nên **kết thúc**: đoạn `[3, 1, 0]`. Nút `4` là nút cuối danh sách nên kết thúc đoạn `[4]`. Tổng `2`.

## 9. Code nộp LeetCode

Ta nộp Cách 2 (`set` + đếm nút cuối đoạn): ngắn gọn, rõ ý, `O(n + m)`.

In [16]:
class Solution:
    def numComponents(self, head: Optional[ListNode], nums: List[int]) -> int:
        s = set(nums)
        ans = 0
        while head:
            if head.val in s and (head.next is None or head.next.val not in s):
                ans += 1
            head = head.next
        return ans


for vals, nums, expected in examples:
    got = Solution().numComponents(build_list(vals), nums)
    print(vals, nums, "->", got, "| kỳ vọng:", expected, "| OK" if got == expected else "| SAI")

[0, 1, 2, 3] [0, 1, 3] -> 2 | kỳ vọng: 2 | OK
[0, 1, 2, 3, 4] [0, 3, 1, 4] -> 2 | kỳ vọng: 2 | OK


## 10. Vì sao đúng

Gọi `T` là dãy nhãn (thuộc / không thuộc `nums`) của các nút theo thứ tự danh sách. Một **thành phần** là một khối `T` liên tiếp **tối đại**.

**(a) Mỗi thành phần có đúng một nút cuối.** Trong khối `T` tối đại, nút cuối cùng của khối có nút sau hoặc không tồn tại, hoặc mang nhãn `F` (nếu mang nhãn `T` thì khối chưa tối đại). Các nút khác của khối có nút sau mang nhãn `T`. Vậy một nút nhãn `T` là "cuối đoạn" **khi và chỉ khi** nó là nút cuối cùng của khối của nó.

**(b) Đếm.** Vì mỗi thành phần ứng với đúng một nút cuối đoạn (và ngược lại), số thành phần bằng số nút nhãn `T` mà nút sau không tồn tại hoặc nhãn `F`. Đó chính là điều kiện của Cách 2.

**(c) Cách 3** đếm **đầu** đoạn: nút nhãn `T` mà nút trước không tồn tại (`prev` ban đầu `False`) hoặc nhãn `F`. Lập luận đối xứng cho thấy mỗi khối có đúng một đầu.

**(d) Cách 4.** Khối `T` có `k` nút có đúng `k - 1` cặp nút liên tiếp bên trong khối (cả hai nút của cặp có nhãn `T`); và không có cặp liên tiếp cùng nhãn `T` nào nằm ở **hai khối khác nhau** (vì giữa hai khối có ít nhất một nút `F`). Do đó `Σ nút - Σ cạnh = Σ_{khối} (k - (k - 1)) = số khối`.

Ta kiểm chứng bằng một lời giải độc lập hoàn toàn: đổi danh sách thành mảng, đánh nhãn rồi dùng `groupby` đếm khối `T`.

In [17]:
def comp_oracle(head, nums):
    s = set(nums)
    flags = [v in s for v in list_to_array(head)]
    return sum(1 for k, _ in groupby(flags) if k)


def random_case(n, rng, p=0.5):
    vals = list(range(n))
    rng.shuffle(vals)                                    # giá trị khác nhau, hoán vị của 0..n-1
    k = max(1, min(n, int(round(n * p))))
    nums = rng.sample(range(n), k)
    rng.shuffle(nums)
    return vals, nums


rng = random.Random(817)
cnt = 0
for _ in range(1000):
    n = rng.randint(1, 30)
    vals, nums = random_case(n, rng, p=rng.choice([0.1, 0.3, 0.5, 0.8, 1.0]))
    head = build_list(vals)
    a = Solution().numComponents(head, nums)
    assert a == comp_oracle(head, nums) == comp_formula(head, nums) == comp_array(head, nums) == comp_bruteforce(head, nums), (vals, nums)
    cnt += 1
print("khớp oracle groupby + 3 cách còn lại trên", cnt, "input ngẫu nhiên")

khớp oracle groupby + 3 cách còn lại trên 1000 input ngẫu nhiên


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1 `in` trên danh sách | `O(n · m)` | `O(1)` |
| 2 `set` + nút cuối đoạn | `O(n + m)` | `O(m)` |
| 3 Mảng `bool` + cờ | `O(n + m)` | `O(n)` |
| 4 Nút trừ cạnh | `O(n + m)` | `O(m)` |

Ta đo thời gian ở kích thước tối đa `n = m = 10^4` (nums là toàn bộ giá trị) và `n = 10^4, m = 5000`.

In [18]:
rng = random.Random(2)
n = 10000
for m_frac in (0.5, 1.0):
    vals, nums = random_case(n, rng, p=m_frac)
    head = build_list(vals)
    res = []
    for fn in (comp_bruteforce, comp_set, comp_array, comp_formula):
        t0 = time.perf_counter()
        r = fn(head, nums)
        res.append((r, round(time.perf_counter() - t0, 4)))
    assert len({r for r, _ in res}) == 1
    print(f"n={n}, m={len(nums)}: đáp án {res[0][0]} | in-list {res[0][1]}s, set {res[1][1]}s, mảng {res[2][1]}s, công thức {res[3][1]}s")

n=10000, m=5000: đáp án 2530 | in-list 0.2356s, set 0.0012s, mảng 0.001s, công thức 0.0011s


n=10000, m=10000: đáp án 1 | in-list 0.3438s, set 0.0021s, mảng 0.0017s, công thức 0.0015s


## 12. Bộ test

Các nhóm test: ví dụ của đề, edge case (một nút, `nums` là toàn bộ danh sách, `nums` chỉ gồm các nút xen kẽ, `nums` ở đầu/cuối), phản ví dụ (danh sách không theo thứ tự giá trị), và đối chiếu **nhiều cách** + oracle `groupby` trên input ngẫu nhiên có seed đúng ràng buộc.

In [19]:
# 1) Ví dụ của đề
for vals, nums, expected in examples:
    assert Solution().numComponents(build_list(vals), nums) == expected

# 2) Edge case
cases = [
    ([0], [0], 1),                                  # một nút thuộc nums
    ([0, 1, 2, 3, 4], [0, 1, 2, 3, 4], 1),          # toàn bộ danh sách: một thành phần
    ([0, 1, 2, 3, 4], [0, 2, 4], 3),                # xen kẽ: ba thành phần
    ([0, 1, 2, 3, 4], [1, 2, 3], 1),                # giữa
    ([0, 1, 2, 3, 4], [0, 4], 2),                   # hai đầu
    ([0, 1, 2, 3, 4], [2], 1),                      # một nút ở giữa
    ([3, 1, 0, 2, 4], [1, 0, 4, 3], 2),             # danh sách không theo thứ tự giá trị
    ([4, 3, 2, 1, 0], [0, 1, 3, 4], 2),             # giảm dần
    ([1, 0], [0, 1], 1),
    ([2, 0, 1], [2, 1], 2),                         # 2 và 1 không liền kề trong danh sách (0 ở giữa)
]
for vals, nums, expected in cases:
    assert Solution().numComponents(build_list(vals), nums) == expected, (vals, nums)
print("ví dụ + edge OK:", len(cases), "ca")

ví dụ + edge OK: 10 ca


In [20]:
# 3) Đối chiếu 5 cách + oracle trên input ngẫu nhiên có seed
rng = random.Random(817)
total = 0
for _ in range(3000):
    n = rng.randint(1, 40)
    vals, nums = random_case(n, rng, p=rng.choice([0.05, 0.2, 0.5, 0.8, 0.95, 1.0]))
    head = build_list(vals)
    e = comp_oracle(head, nums)
    for fn in (comp_bruteforce, comp_set, comp_array, comp_formula, Solution().numComponents):
        assert fn(head, nums) == e, (fn, vals, nums)
    total += 1
print("3000 input ngẫu nhiên x 5 cách: OK")

# 4) Giới hạn tối đa n = 10^4 (không dùng Cách 1)
rng = random.Random(10000)
for p in (0.1, 0.5, 0.9, 1.0):
    vals, nums = random_case(10000, rng, p)
    head = build_list(vals)
    assert Solution().numComponents(head, nums) == comp_oracle(head, nums) == comp_formula(head, nums) == comp_array(head, nums)
print("n = 10^4 OK")

# 5) Thứ tự nums không ảnh hưởng, danh sách đầu vào không bị sửa
vals, nums = random_case(200, random.Random(5), 0.5)
head = build_list(vals)
a = Solution().numComponents(head, nums)
nums2 = nums[:]; random.Random(6).shuffle(nums2)
assert Solution().numComponents(head, nums2) == a
assert list_to_array(head) == vals
print("thứ tự nums không ảnh hưởng; danh sách không bị sửa")

3000 input ngẫu nhiên x 5 cách: OK
n = 10^4 OK
thứ tự nums không ảnh hưởng; danh sách không bị sửa


## 13. Các lỗi người mới hay gặp

Ta cài vài hàm sai cố ý rồi tìm input nhỏ nhất làm chúng sai bằng `find_mismatch` (so với oracle `groupby`).

In [21]:
def find_mismatch(fn, trials=3000, seed=9):
    rng = random.Random(seed)
    best = None
    for _ in range(trials):
        n = rng.randint(1, 7)
        vals, nums = random_case(n, rng, p=rng.choice([0.4, 0.6, 0.8]))
        head = build_list(vals)
        try:
            ok = fn(head, nums) == comp_oracle(head, nums)
        except Exception:
            ok = False
        if not ok and (best is None or len(vals) < len(best[0])):
            best = (vals, nums)
    return best

### Lỗi 1: đếm số nút thuộc `nums` thay vì số thành phần

Đề hỏi số **đoạn liên tiếp**, không phải số phần tử.

In [22]:
def wrong_count_nodes(head, nums):
    s = set(nums)
    ans = 0
    while head:
        if head.val in s:
            ans += 1                              # SAI: đếm từng nút
        head = head.next
    return ans

print("sai :", wrong_count_nodes(build_list([0, 1, 2, 3]), [0, 1, 3]), "| đúng: 2")
print("input nhỏ nhất làm sai:", find_mismatch(wrong_count_nodes))

sai : 3 | đúng: 2
input nhỏ nhất làm sai: ([1, 0], [1, 0])


**Vì sao sai:** nút `0` và `1` thuộc **cùng một** thành phần nhưng bị đếm hai lần. **Code đúng:** chỉ đếm **một lần cho mỗi đoạn** (ở nút cuối hoặc nút đầu của đoạn).

### Lỗi 2: truy cập `head.next.val` khi `head.next` là `None`

Điều kiện "nút sau không thuộc `nums`" bị viết gọn thành `head.next.val not in s`. Ở nút cuối danh sách `head.next` là `None`, gây `AttributeError`.

In [23]:
def wrong_next_none(head, nums):
    s = set(nums)
    ans = 0
    while head:
        if head.val in s and head.next.val not in s:      # SAI: head.next có thể là None
            ans += 1
        head = head.next
    return ans

try:
    print(wrong_next_none(build_list([0, 1, 2, 3]), [0, 1, 3]))
except AttributeError as e:
    print("AttributeError:", e)
print("input nhỏ nhất làm sai (hoặc lỗi):", find_mismatch(wrong_next_none))

AttributeError: 'NoneType' object has no attribute 'val'


input nhỏ nhất làm sai (hoặc lỗi): ([0], [0])


**Vì sao sai:** nút cuối danh sách không có nút sau; nếu nút cuối thuộc `nums` thì nó **là** cuối đoạn nhưng ta không được đọc `.val` của `None`. **Code đúng:** `head.next is None or head.next.val not in s` (kiểm tra `None` **trước** nhờ tính "đoản mạch" của `or`).

### Lỗi 3: quên đặt lại cờ khi gặp nút không thuộc `nums`

Dùng cờ `in_run` nhưng chỉ đặt `True` mà không bao giờ đặt lại `False`, nên các đoạn sau đoạn đầu không được đếm.

In [24]:
def wrong_flag_not_reset(head, nums):
    s = set(nums)
    ans = 0
    in_run = False
    while head:
        if head.val in s:
            if not in_run:
                ans += 1
                in_run = True
        # SAI: thiếu else: in_run = False
        head = head.next
    return ans

print("sai :", wrong_flag_not_reset(build_list([0, 1, 2, 3]), [0, 1, 3]), "| đúng: 2")
print("input nhỏ nhất làm sai:", find_mismatch(wrong_flag_not_reset))

sai : 1 | đúng: 2
input nhỏ nhất làm sai: ([0, 1, 2], [2, 0])


**Vì sao sai:** sau khi gặp nút `2` không thuộc `nums`, cờ vẫn `True` nên nút `3` bị coi là tiếp tục đoạn cũ; chỉ đếm được `1`. **Code đúng:** `else: in_run = False` (hoặc `prev = cur` ở mọi vòng như Cách 3).

### Lỗi 4: nhầm "liên tiếp trong danh sách" với "liên tiếp về giá trị"

Một số bạn đọc đề thành "đếm số dãy số nguyên liên tiếp trong `nums`". Với danh sách hoán vị bất kỳ, hai giá trị `v` và `v + 1` có thể ở rất xa nhau trong danh sách.

In [25]:
def wrong_by_value(head, nums):
    s = sorted(nums)
    ans = 0
    for i, v in enumerate(s):
        if i == 0 or s[i - 1] != v - 1:               # SAI: đếm khối giá trị liên tiếp, bỏ qua thứ tự danh sách
            ans += 1
    return ans

bad = find_mismatch(wrong_by_value)
print("input nhỏ nhất làm sai:", bad)
vals, nums = bad
print("danh sách:", vals, "| nums:", nums, "| sai:", wrong_by_value(build_list(vals), nums), "| đúng:", Solution().numComponents(build_list(vals), nums))
print("ví dụ của đề vẫn đúng (tình cờ vì danh sách đã sắp xếp):", [wrong_by_value(build_list(v), n) == e for v, n, e in examples])

input nhỏ nhất làm sai: ([0, 2, 1], [0, 2])
danh sách: [0, 2, 1] | nums: [0, 2] | sai: 2 | đúng: 1
ví dụ của đề vẫn đúng (tình cờ vì danh sách đã sắp xếp): [True, True]


**Vì sao sai:** ví dụ của đề có danh sách tăng dần nên hai cách hiểu trùng nhau, che giấu lỗi. Khi danh sách là hoán vị bất kỳ, "liên tiếp" phải hiểu theo **vị trí trong danh sách**. **Code đúng:** duyệt danh sách theo `next`, không sắp xếp `nums`.

## 14. Mẹo & nhận dạng pattern

- Thấy "đếm số đoạn/nhóm **liên tiếp** thỏa điều kiện trong một dãy" → gắn **nhãn T/F** rồi đếm số khối `T`: bằng số lần chuyển `F -> T` (đầu đoạn) hoặc `T -> F` (cuối đoạn).
- Có hai cách đếm tương đương: **đầu đoạn** (cần nhớ phần tử trước, dùng cờ) hoặc **cuối đoạn** (nhìn phần tử sau, dùng `head.next`).
- Với danh sách liên kết, "nhìn phần tử sau" rất tự nhiên: nhớ kiểm tra `head.next is None` **trước**.
- Hay dùng `set` để tra cứu thành viên: đổi `list` thành `set` một lần là cách tối ưu quen thuộc.
- Công thức `số đỉnh - số cạnh` (với đồ thị là rừng/các đường đi) đếm số thành phần mà không cần cờ.
- Nhớ đối chiếu với một **oracle độc lập** (đổi sang mảng + `groupby`) để bắt lỗi logic.
- Test đừng quên: một nút, `nums` là tất cả, `nums` xen kẽ, danh sách hoán vị bất kỳ.

## 15. Biến thể

### 15.1. Trả về chính các thành phần

Thay vì đếm, trả về danh sách các thành phần (mỗi thành phần là danh sách giá trị theo thứ tự trong danh sách liên kết).

In [26]:
def components_lists(head, nums):
    s = set(nums)
    res, cur = [], []
    while head:
        if head.val in s:
            cur.append(head.val)
        elif cur:
            res.append(cur)
            cur = []
        head = head.next
    if cur:
        res.append(cur)
    return res

print(components_lists(build_list([3, 1, 0, 2, 4]), [1, 0, 4, 3]))      # [[3, 1, 0], [4]]
rng = random.Random(151)
for _ in range(300):
    vals, nums = random_case(rng.randint(1, 25), rng, rng.choice([0.3, 0.6, 0.9]))
    assert len(components_lists(build_list(vals), nums)) == comp_oracle(build_list(vals), nums)
print("số danh sách trả về khớp số thành phần")

[[3, 1, 0], [4]]
số danh sách trả về khớp số thành phần


### 15.2. Độ dài thành phần dài nhất

Cùng khuôn mẫu: đếm độ dài của khối `T` hiện tại và lấy `max`.

In [27]:
def longest_component(head, nums):
    s = set(nums)
    best = cur = 0
    while head:
        cur = cur + 1 if head.val in s else 0
        best = max(best, cur)
        head = head.next
    return best

print(longest_component(build_list([3, 1, 0, 2, 4]), [1, 0, 4, 3]))     # 3

3


### 15.3. Đếm số thành phần của các nút **không** thuộc `nums`

Đổi điều kiện thành `head.val not in s`. Hai số này liên hệ: nếu có `a` thành phần thuộc `nums` và `b` thành phần không thuộc thì `|a - b| <= 1` (hai loại xen kẽ nhau). Ta kiểm tra tính chất này bằng thí nghiệm.

In [28]:
def comp_complement(head, nums):
    s = set(nums)
    ans = 0
    while head:
        if head.val not in s and (head.next is None or head.next.val in s):
            ans += 1
        head = head.next
    return ans

rng = random.Random(153)
for _ in range(500):
    vals, nums = random_case(rng.randint(2, 30), rng, rng.choice([0.3, 0.6, 0.9]))
    head = build_list(vals)
    assert abs(comp_oracle(head, nums) - comp_complement(head, nums)) <= 1
print("|a - b| <= 1 đúng trên 500 input")

|a - b| <= 1 đúng trên 500 input


### 15.4. Bài liên quan

- LeetCode 876 Middle of the Linked List, 206 Reverse Linked List: duyệt/di chuyển trên danh sách.
- LeetCode 128 Longest Consecutive Sequence: đoạn liên tiếp về **giá trị** (khác bài này).
- LeetCode 547 Number of Provinces, 323 Number of Connected Components in an Undirected Graph: thành phần liên thông tổng quát (cần DSU/DFS).
- LeetCode 485 Max Consecutive Ones: đếm độ dài đoạn `1` liên tiếp.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 `in` trên danh sách | cờ `in_run` + tra `list` | `O(n · m)` | `O(1)` | chỉ để hiểu đề |
| 2 `set` + cuối đoạn | tra `set`, đếm nút cuối đoạn | `O(n + m)` | `O(m)` | **cách nộp**, ngắn gọn |
| 3 Mảng `bool` + cờ | chỉ mục theo giá trị, đếm đầu đoạn | `O(n + m)` | `O(n)` | khi giá trị nhỏ và đã biết giới hạn |
| 4 Nút trừ cạnh | `Σ nút - Σ cạnh` | `O(n + m)` | `O(m)` | khi muốn công thức không cờ |

## 17. Bài tập vận dụng

Làm lần lượt từ dễ đến khó. Gợi ý ở mục 18, đáp án ở mục 19.

### Bài 1 (đoán output)

Hàm `labels(head, nums)` (mục 4) trả dãy nhãn `T/F` theo thứ tự danh sách. Đoán kết quả cho `head = [2, 0, 3, 1, 4]`, `nums = [0, 1, 2, 4]` (danh sách 5 ký tự `"T"`/`"F"`) rồi điền `ex1_answer`. **Không** chạy trước khi đoán.

In [29]:
ex1_head, ex1_nums = [2, 0, 3, 1, 4], [0, 1, 2, 4]
ex1_answer = None       # ví dụ ["T", "T", "T", "T", "T"]  <- chỉ là mẫu định dạng

### Bài 2 (tính tay)

Tính tay số thành phần cho ba trường hợp (danh sách, `nums`): `([4, 2, 0, 1, 3], [4, 0, 1, 3])`, `([0, 1, 2, 3, 4, 5], [0, 2, 3, 5])`, `([5, 4, 3, 2, 1, 0], [1, 2, 3])`. Điền `ex2_answer` là danh sách 3 số.

In [30]:
ex2_cases = [([4, 2, 0, 1, 3], [4, 0, 1, 3]), ([0, 1, 2, 3, 4, 5], [0, 2, 3, 5]), ([5, 4, 3, 2, 1, 0], [1, 2, 3])]
ex2_answer = None       # ví dụ [0, 0, 0]

### Bài 3 (điền TODO)

Hoàn thành `ex3_components`: hai chỗ `TODO` là điều kiện "nút này là cuối đoạn". Cell này **không gọi** hàm.

In [31]:
def ex3_components(head, nums):
    s = set(nums)
    ans = 0
    while head:
        # TODO 1: điều kiện đầu: head.val có thuộc s không?
        # TODO 2: điều kiện nút sau: không tồn tại hoặc không thuộc s
        if False:
            ans += 1
        head = head.next
    return ans

### Bài 4 (tự viết lại)

Tự viết **không nhìn** hàm `ex4_rewrite(head, nums)` bằng **Cách 4** (`số nút - số cạnh`). Để `pass` nếu chưa làm.

In [32]:
def ex4_rewrite(head, nums):
    pass

### Bài 5 (biến thể / mini challenge)

Viết `ex5_longest_gap(head, nums)`: trả về độ dài của **đoạn nút KHÔNG thuộc `nums`** liên tiếp **dài nhất** (0 nếu mọi nút đều thuộc `nums`). Ví dụ `head = [0,1,2,3,4,5]`, `nums = [0, 5]` cho `4`.

In [33]:
def ex5_longest_gap(head, nums):
    pass

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: đọc danh sách từ trái sang phải, nhãn `T` nếu giá trị thuộc `nums`.
- Mức 2: `nums = {0, 1, 2, 4}`, nên chỉ `3` thuộc nhãn `F`.
- Mức 3: `["T", "T", "F", "T", "T"]`.

**Bài 2**
- Mức 1: viết dãy nhãn `T/F` rồi đếm khối `T`.
- Mức 2: ca 1: `4, 2, 0, 1, 3` với `nums = {4, 0, 1, 3}` cho nhãn `T F T T T` (hai khối); ca 2 cho `T F T T F T`; ca 3 cho `F F T T T F`.
- Mức 3: số khối `T` lần lượt là `2, 3, 1`, tức đáp án `[2, 3, 1]`.

**Bài 3**
- Mức 1: nút phải thuộc `nums` **và** là cuối đoạn.
- Mức 2: TODO 1: `head.val in s`; TODO 2: `head.next is None or head.next.val not in s`.
- Mức 3: `if head.val in s and (head.next is None or head.next.val not in s):`.

**Bài 4**
- Mức 1: đếm hai biến `nodes` và `edges`.
- Mức 2: cạnh khi cả nút hiện tại và nút sau thuộc `s`.
- Mức 3: trả `nodes - edges`.

**Bài 5**
- Mức 1: giống đếm độ dài đoạn liên tiếp, nhưng điều kiện đảo lại.
- Mức 2: `cur = cur + 1 if head.val not in s else 0`.
- Mức 3: cập nhật `best = max(best, cur)` ở mỗi nút.

## 19. Đáp án tham khảo

### Đáp án Bài 1

In [34]:
ex1_solution = ["T", "T", "F", "T", "T"]
assert labels(build_list(ex1_head), ex1_nums) == ex1_solution
print("Bài 1 OK:", ex1_solution)

Bài 1 OK: ['T', 'T', 'F', 'T', 'T']


### Đáp án Bài 2

In [35]:
ex2_solution = [2, 3, 1]
for (vals, nums), e in zip(ex2_cases, ex2_solution):
    assert Solution().numComponents(build_list(vals), nums) == e == comp_oracle(build_list(vals), nums), (vals, nums)
    print(vals, nums, "->", "".join(labels(build_list(vals), nums)), "->", e)
print("Bài 2 OK:", ex2_solution)

[4, 2, 0, 1, 3] [4, 0, 1, 3] -> TFTTT -> 2
[0, 1, 2, 3, 4, 5] [0, 2, 3, 5] -> TFTTFT -> 3
[5, 4, 3, 2, 1, 0] [1, 2, 3] -> FFTTTF -> 1
Bài 2 OK: [2, 3, 1]


### Đáp án Bài 3

In [36]:
def ex3_solution(head, nums):
    s = set(nums)
    ans = 0
    while head:
        if head.val in s and (head.next is None or head.next.val not in s):
            ans += 1
        head = head.next
    return ans

rng = random.Random(3)
for _ in range(300):
    vals, nums = random_case(rng.randint(1, 30), rng, rng.choice([0.2, 0.5, 0.9]))
    assert ex3_solution(build_list(vals), nums) == comp_oracle(build_list(vals), nums)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [37]:
def ex4_solution(head, nums):
    s = set(nums)
    nodes = edges = 0
    while head:
        if head.val in s:
            nodes += 1
            if head.next is not None and head.next.val in s:
                edges += 1
        head = head.next
    return nodes - edges

rng = random.Random(4)
for _ in range(300):
    vals, nums = random_case(rng.randint(1, 30), rng, rng.choice([0.2, 0.5, 0.9]))
    assert ex4_solution(build_list(vals), nums) == comp_oracle(build_list(vals), nums)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [38]:
def ex5_solution(head, nums):
    s = set(nums)
    best = cur = 0
    while head:
        cur = cur + 1 if head.val not in s else 0
        best = max(best, cur)
        head = head.next
    return best

assert ex5_solution(build_list([0, 1, 2, 3, 4, 5]), [0, 5]) == 4
assert ex5_solution(build_list([0, 1, 2]), [0, 1, 2]) == 0
rng = random.Random(5)
for _ in range(300):
    vals, nums = random_case(rng.randint(1, 30), rng, rng.choice([0.2, 0.5, 0.9]))
    s = set(nums)
    flags = [v not in s for v in vals]
    expected = max([len(list(g)) for k, g in groupby(flags) if k], default=0)       # oracle độc lập
    assert ex5_solution(build_list(vals), nums) == expected
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def numComponents(self, head: Optional[ListNode], nums: List[int]) -> int:
        s = set(nums)                                          # tra cứu O(1)
        ans = 0
        while head:
            # nút thuộc nums và là CUỐI đoạn: nút sau không có hoặc không thuộc nums
            if head.val in s and (head.next is None or head.next.val not in s):
                ans += 1
            head = head.next
        return ans
```

Câu thần chú: **"đổi nums sang set, đếm nút cuối đoạn, nhớ kiểm tra next là None trước."**

### Checklist trước khi submit

- [ ] `nums` đã đổi sang `set` (không dùng `in list` cho từng nút).
- [ ] Đếm **đoạn**, không đếm nút.
- [ ] Kiểm tra `head.next is None` **trước** khi đọc `head.next.val`.
- [ ] "Liên tiếp" theo **vị trí trong danh sách**, không phải theo giá trị.
- [ ] Nếu dùng cờ: đặt lại `False` khi gặp nút không thuộc `nums`.
- [ ] Test: một nút, `nums` là tất cả, xen kẽ, danh sách hoán vị bất kỳ.
- [ ] Skeleton đúng: `class Solution`, method `numComponents(self, head: Optional[ListNode], nums: List[int]) -> int`.